# =============================================================================
# TITAN-FND ABLATION STUDY — WELFake Dataset
# Run this on Kaggle with GPU enabled (T4 or P100)
# =============================================================================
# Ablation variants:
#  A0 — Full TITAN-FND            ← proposed model (should win)
#  A1 — No Orthogonal Loss        ← remove diversity constraint
#  A2 — No Focal Loss             ← plain CrossEntropy
#  A3 — No Gated Fusion           ← simple mean of three streams
#  A4 — Semantic Stream Only      ← single head
#  A5 — Emotional Stream Only     ← single head
#  A6 — Syntactic Stream Only     ← single head
#  A7 — No CLS Path               ← fused only, no CLS concat
#  A8 — Plain DeBERTa Baseline    ← CLS → MLP, no streams at all
# =============================================================================

## Cell 1: Install

In [ ]:
# %%capture
# !pip install transformers==4.40.0 accelerate sentencepiece protobuf -q


## Cell 2: Imports & Seed

In [ ]:
import os, re, random, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix,
                             precision_score, recall_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


## Cell 3: Config

In [ ]:
MODEL_NAME  = 'microsoft/deberta-v3-base'
MAX_LEN     = 64
BATCH_SIZE  = 16
EPOCHS      = 10
LR_ENC      = 5e-6
LR_HEAD     = 5e-5
WARMUP      = 0.15
PATIENCE    = 3
ORTHO_W     = 0.1
DROPOUT     = 0.5

DATA_PATH = '/kaggle/input/datasets/saurabhshahane/fake-news-classification/WELFake_Dataset.csv'
OUT_DIR   = '/kaggle/working'


## Cell 4: Load & Clean WELFake

In [ ]:
df = pd.read_csv(DATA_PATH, engine='python', on_bad_lines='skip')
df.columns = df.columns.str.strip().str.lower()
df = df[['title', 'label']].copy()
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
df['title'] = df['title'].fillna('').astype(str)

def clean_title(text):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

df['title'] = df['title'].apply(clean_title)
df = df[df['title'].str.len() > 5].reset_index(drop=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Total  : {len(df)}')
print(f'Label 0 (Fake): {(df.label==0).sum()}')
print(f'Label 1 (Real): {(df.label==1).sum()}')


## Cell 5: Train / Val / Test Split

In [ ]:
X = df['title'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)

print(f'Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')


## Cell 6: Tokenizer & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class TitleDataset(Dataset):
    def __init__(self, titles, labels, tokenizer, max_len):
        self.titles    = titles
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.titles)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.titles[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'token_type_ids' : enc.get(
                'token_type_ids',
                torch.zeros(self.max_len, dtype=torch.long)
            ).squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = TitleDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = TitleDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = TitleDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f'Batches — train:{len(train_dl)} val:{len(val_dl)} test:{len(test_dl)}')


## Cell 7: Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = (1 - pt) ** self.gamma * ce_loss
        if self.alpha is not None:
            focal = self.alpha[targets] * focal
        return focal.mean() if self.reduction == 'mean' else focal.sum()

class_counts = np.bincount(y_train)
total_count  = class_counts.sum()
alpha_vals   = torch.tensor(
    [total_count / (2 * class_counts[0]),
     total_count / (2 * class_counts[1])],
    dtype=torch.float).to(DEVICE)
print(f'Alpha — class0:{alpha_vals[0]:.4f}  class1:{alpha_vals[1]:.4f}')


## Cell 8: Building Blocks

In [ ]:
class TripleStreamAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        d = hidden_size // 2
        self.sem_proj = nn.Linear(hidden_size, d)
        self.emo_proj = nn.Linear(hidden_size, d)
        self.syn_proj = nn.Linear(hidden_size, d)
        self.sem_attn = nn.Linear(d, 1)
        self.emo_attn = nn.Linear(d, 1)
        self.syn_attn = nn.Linear(d, 1)
        self.d = d

    def _attend(self, proj, attn, hidden, mask):
        h       = torch.tanh(proj(hidden))
        scores  = attn(h).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * h).sum(dim=1)

    def forward(self, hidden, mask):
        sem = self._attend(self.sem_proj, self.sem_attn, hidden, mask)
        emo = self._attend(self.emo_proj, self.emo_attn, hidden, mask)
        syn = self._attend(self.syn_proj, self.syn_attn, hidden, mask)
        return sem, emo, syn


class SingleStreamAttention(nn.Module):
    """Used in A4/A5/A6."""
    def __init__(self, hidden_size):
        super().__init__()
        d = hidden_size // 2
        self.proj = nn.Linear(hidden_size, d)
        self.attn = nn.Linear(d, 1)
        self.d    = d

    def forward(self, hidden, mask):
        h       = torch.tanh(self.proj(hidden))
        scores  = self.attn(h).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (weights.unsqueeze(-1) * h).sum(dim=1)


class GatedFusion(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.gate = nn.Linear(d * 3, 3)

    def forward(self, sem, emo, syn):
        concat = torch.cat([sem, emo, syn], dim=-1)
        gates  = torch.sigmoid(self.gate(concat))
        g_s, g_e, g_n = gates[:, 0:1], gates[:, 1:2], gates[:, 2:3]
        fused  = g_s * sem + g_e * emo + g_n * syn
        return fused, gates


def make_classifier(in_dim, d, dropout):
    return nn.Sequential(
        nn.Linear(in_dim, d),
        nn.LayerNorm(d),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(d, d // 2),
        nn.LayerNorm(d // 2),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(d // 2, 2)
    )


def init_weights(modules):
    for m in modules:
        for p in m.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)


def freeze_encoder(model, keep_top_n=3):
    enc = model.encoder
    # freeze embeddings
    for p in enc.embeddings.parameters():
        p.requires_grad = False
    total  = len(enc.encoder.layer)
    freeze = total - keep_top_n
    for i, layer in enumerate(enc.encoder.layer):
        if i < freeze:
            for p in layer.parameters():
                p.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f'  Params: {trainable/1e6:.1f}M trainable / {total_p/1e6:.1f}M total')


## Cell 9: All 9 Model Variants

In [ ]:
# A0 ── Full TITAN-FND (your proposed model, exact replica of WELFake notebook)
class A0_TITANFND_FULL(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder      = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size
        d = H // 2
        self.triple_attn  = TripleStreamAttention(H)
        self.gated_fusion = GatedFusion(d)
        self.cls_proj     = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier   = make_classifier(d * 2, d, dropout)
        init_weights([self.triple_attn, self.gated_fusion,
                      self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden = out.last_hidden_state
        cls    = hidden[:, 0, :]
        sem, emo, syn     = self.triple_attn(hidden, attention_mask)
        fused, gates      = self.gated_fusion(sem, emo, syn)
        cls_out           = self.cls_proj(cls)
        combined          = torch.cat([fused, cls_out], dim=-1)
        logits            = self.classifier(combined)
        return logits, gates, sem, emo, syn


# A1 ── No Orthogonal Loss (same arch, ortho disabled in training loop)
class A1_NO_ORTHO(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder      = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.triple_attn  = TripleStreamAttention(H)
        self.gated_fusion = GatedFusion(d)
        self.cls_proj     = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier   = make_classifier(d * 2, d, dropout)
        init_weights([self.triple_attn, self.gated_fusion,
                      self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden = out.last_hidden_state; cls = hidden[:, 0, :]
        sem, emo, syn = self.triple_attn(hidden, attention_mask)
        fused, gates  = self.gated_fusion(sem, emo, syn)
        cls_out       = self.cls_proj(cls)
        logits        = self.classifier(torch.cat([fused, cls_out], dim=-1))
        return logits, gates, sem, emo, syn


# A2 ── No Focal Loss (plain CE in training, same arch)
class A2_NO_FOCAL(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder      = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.triple_attn  = TripleStreamAttention(H)
        self.gated_fusion = GatedFusion(d)
        self.cls_proj     = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier   = make_classifier(d * 2, d, dropout)
        init_weights([self.triple_attn, self.gated_fusion,
                      self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden = out.last_hidden_state; cls = hidden[:, 0, :]
        sem, emo, syn = self.triple_attn(hidden, attention_mask)
        fused, gates  = self.gated_fusion(sem, emo, syn)
        cls_out       = self.cls_proj(cls)
        logits        = self.classifier(torch.cat([fused, cls_out], dim=-1))
        return logits, gates, sem, emo, syn


# A3 ── No Gated Fusion (simple mean of three streams)
class A3_NO_GATES(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder     = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.triple_attn = TripleStreamAttention(H)
        self.cls_proj    = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier  = make_classifier(d * 2, d, dropout)
        init_weights([self.triple_attn, self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden  = out.last_hidden_state; cls = hidden[:, 0, :]
        sem, emo, syn = self.triple_attn(hidden, attention_mask)
        fused   = (sem + emo + syn) / 3.0   # equal weight, no learning
        gates   = torch.ones(input_ids.size(0), 3,
                             device=input_ids.device) / 3.0
        cls_out = self.cls_proj(cls)
        logits  = self.classifier(torch.cat([fused, cls_out], dim=-1))
        return logits, gates, sem, emo, syn


# A4 ── Semantic Stream Only
class A4_SEM_ONLY(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.sem        = SingleStreamAttention(H)
        self.cls_proj   = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier = make_classifier(d * 2, d, dropout)
        init_weights([self.sem, self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden  = out.last_hidden_state; cls = hidden[:, 0, :]
        sem     = self.sem(hidden, attention_mask)
        cls_out = self.cls_proj(cls)
        logits  = self.classifier(torch.cat([sem, cls_out], dim=-1))
        dummy   = torch.zeros_like(sem)
        gates   = torch.ones(input_ids.size(0), 3,
                             device=input_ids.device) / 3.0
        return logits, gates, sem, dummy, dummy


# A5 ── Emotional Stream Only
class A5_EMO_ONLY(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.emo        = SingleStreamAttention(H)
        self.cls_proj   = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier = make_classifier(d * 2, d, dropout)
        init_weights([self.emo, self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden  = out.last_hidden_state; cls = hidden[:, 0, :]
        emo     = self.emo(hidden, attention_mask)
        cls_out = self.cls_proj(cls)
        logits  = self.classifier(torch.cat([emo, cls_out], dim=-1))
        dummy   = torch.zeros_like(emo)
        gates   = torch.ones(input_ids.size(0), 3,
                             device=input_ids.device) / 3.0
        return logits, gates, dummy, emo, dummy


# A6 ── Syntactic Stream Only
class A6_SYN_ONLY(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.syn        = SingleStreamAttention(H)
        self.cls_proj   = nn.Sequential(nn.Linear(H, d), nn.Tanh())
        self.classifier = make_classifier(d * 2, d, dropout)
        init_weights([self.syn, self.cls_proj, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden  = out.last_hidden_state; cls = hidden[:, 0, :]
        syn     = self.syn(hidden, attention_mask)
        cls_out = self.cls_proj(cls)
        logits  = self.classifier(torch.cat([syn, cls_out], dim=-1))
        dummy   = torch.zeros_like(syn)
        gates   = torch.ones(input_ids.size(0), 3,
                             device=input_ids.device) / 3.0
        return logits, gates, dummy, dummy, syn


# A7 ── No CLS Path (fused stream only, no CLS concat)
class A7_NO_CLS(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder      = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.triple_attn  = TripleStreamAttention(H)
        self.gated_fusion = GatedFusion(d)
        self.classifier   = make_classifier(d, d, dropout)   # d not d*2
        init_weights([self.triple_attn, self.gated_fusion, self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        hidden = out.last_hidden_state
        sem, emo, syn  = self.triple_attn(hidden, attention_mask)
        fused, gates   = self.gated_fusion(sem, emo, syn)
        logits         = self.classifier(fused)   # no CLS
        return logits, gates, sem, emo, syn


# A8 ── Plain DeBERTa (CLS → MLP only, zero streams)
class A8_PLAIN_DEBERTA(nn.Module):
    def __init__(self, model_name, dropout=0.5):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        H = self.encoder.config.hidden_size; d = H // 2
        self.classifier = make_classifier(H, d, dropout)
        init_weights([self.classifier])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)
        cls    = out.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        d      = cls.size(-1) // 2
        dummy  = torch.zeros(input_ids.size(0), d, device=input_ids.device)
        gates  = torch.ones(input_ids.size(0), 3,  device=input_ids.device) / 3.0
        return logits, gates, dummy, dummy, dummy


## Cell 10: Orthogonal Loss

In [ ]:
def orthogonal_loss(sem, emo, syn):
    s = F.normalize(sem, dim=-1)
    e = F.normalize(emo, dim=-1)
    n = F.normalize(syn, dim=-1)
    return ((s * e).sum(-1).abs().mean() +
            (s * n).sum(-1).abs().mean() +
            (e * n).sum(-1).abs().mean()) / 3.0


## Cell 11: Train & Evaluate Helpers

In [ ]:
def build_opt_sched(model, n_batches, epochs):
    enc_p  = [p for p in model.encoder.parameters() if p.requires_grad]
    head_p = [p for n, p in model.named_parameters()
               if p.requires_grad and not n.startswith('encoder')]
    opt = torch.optim.AdamW([
        {'params': enc_p,  'lr': LR_ENC,  'weight_decay': 0.01},
        {'params': head_p, 'lr': LR_HEAD, 'weight_decay': 0.001}
    ], eps=1e-8)
    total   = n_batches * epochs
    warmup  = int(total * WARMUP)
    sched   = get_cosine_schedule_with_warmup(opt, warmup, total)
    return opt, sched


def train_epoch(model, loader, opt, sched, crit, device, use_ortho):
    model.train()
    total_loss = correct = total = 0
    for batch in loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        ttype  = batch['token_type_ids'].to(device)
        labels = batch['label'].to(device)
        opt.zero_grad()
        logits, _, sem, emo, syn = model(ids, mask, ttype)
        loss = crit(logits, labels)
        if use_ortho:
            loss = loss + ORTHO_W * orthogonal_loss(sem, emo, syn)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(-1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, crit, device, threshold=0.5):
    model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    for batch in loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        ttype  = batch['token_type_ids'].to(device)
        labels = batch['label'].to(device)
        logits, _, _, _, _ = model(ids, mask, ttype)
        total_loss += crit(logits, labels).item() * labels.size(0)
        probs = torch.softmax(logits, dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs >= threshold).astype(int)
    acc    = accuracy_score(labels, preds)
    f1     = f1_score(labels, preds, average='weighted')
    return total_loss / len(loader.dataset), acc, f1, preds, labels, probs


def tune_threshold(model, loader, crit, device):
    _, _, _, _, labels, probs = evaluate(model, loader, crit, device, 0.5)
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.25, 0.66, 0.01):
        preds = (probs >= t).astype(int)
        f1    = f1_score(labels, preds, average='weighted')
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return round(float(best_t), 2)


## Cell 12: Train One Variant

In [ ]:
def train_variant(name, model_cls, use_ortho, use_focal):
    print(f'\n{"="*62}')
    print(f'  {name}')
    print(f'{"="*62}')

    model = model_cls(MODEL_NAME, dropout=DROPOUT).to(DEVICE)
    freeze_encoder(model, keep_top_n=3)

    crit  = (FocalLoss(alpha=alpha_vals, gamma=2.0)
             if use_focal else nn.CrossEntropyLoss())

    opt, sched = build_opt_sched(model, len(train_dl), EPOCHS)

    best_f1, no_imp = 0.0, 0
    ckpt = f'{OUT_DIR}/{name}_best.pt'
    hist = {'tr_loss': [], 'vl_loss': [], 'tr_acc': [], 'vl_acc': [], 'vl_f1': []}

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc          = train_epoch(model, train_dl, opt, sched,
                                               crit, DEVICE, use_ortho)
        vl_loss, vl_acc, vl_f1, _, _, _ = evaluate(model, val_dl, crit,
                                                    DEVICE, 0.5)
        hist['tr_loss'].append(tr_loss)
        hist['vl_loss'].append(vl_loss)
        hist['tr_acc'].append(tr_acc)
        hist['vl_acc'].append(vl_acc)
        hist['vl_f1'].append(vl_f1)

        tag = ' ◀ best' if vl_f1 > best_f1 else ''
        print(f'  Ep {epoch:02d}/{EPOCHS} | '
              f'tr_acc:{tr_acc:.4f}  vl_acc:{vl_acc:.4f}  vl_f1:{vl_f1:.4f}{tag}')

        if vl_f1 > best_f1:
            best_f1 = vl_f1; no_imp = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break

    elapsed = (time.time() - t0) / 60
    print(f'  Best val F1: {best_f1:.4f}  |  Time: {elapsed:.1f} min')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, hist


## Cell 13: Run All Variants

In [ ]:
# (name, class, use_ortho, use_focal)
VARIANTS = [
    ('A0_TITANFND_FULL', A0_TITANFND_FULL, True,  True),
    ('A1_NO_ORTHO',      A1_NO_ORTHO,      False, True),
    ('A2_NO_FOCAL',      A2_NO_FOCAL,      True,  False),
    ('A3_NO_GATES',      A3_NO_GATES,      True,  True),
    ('A4_SEM_ONLY',      A4_SEM_ONLY,      True,  True),
    ('A5_EMO_ONLY',      A5_EMO_ONLY,      True,  True),
    ('A6_SYN_ONLY',      A6_SYN_ONLY,      True,  True),
    ('A7_NO_CLS',        A7_NO_CLS,        True,  True),
    ('A8_PLAIN_DEBERTA', A8_PLAIN_DEBERTA, False, True),
]

trained_models = {}
all_histories  = {}

for name, cls, use_ortho, use_focal in VARIANTS:
    model, hist = train_variant(name, cls, use_ortho, use_focal)
    trained_models[name] = model
    all_histories[name]  = hist
    torch.cuda.empty_cache()


## Cell 14: Test Evaluation — All Variants

In [ ]:
print('\n\n' + '='*62)
print('  FINAL TEST RESULTS — ALL VARIANTS')
print('='*62)

final_results = []

for name, _, _, use_focal in VARIANTS:
    model = trained_models[name]
    crit  = (FocalLoss(alpha=alpha_vals, gamma=2.0)
             if use_focal else nn.CrossEntropyLoss())

    best_t = tune_threshold(model, val_dl, crit, DEVICE)
    _, test_acc, test_f1, preds, labels, _ = evaluate(
        model, test_dl, crit, DEVICE, best_t)

    report = classification_report(
        labels, preds,
        target_names=['Fake', 'Real'],
        output_dict=True, zero_division=0)

    prec_fake   = report['Fake']['precision']
    rec_fake    = report['Fake']['recall']
    f1_fake     = report['Fake']['f1-score']
    prec_real   = report['Real']['precision']
    rec_real    = report['Real']['recall']
    f1_real     = report['Real']['f1-score']

    final_results.append({
        'Variant'    : name,
        'Threshold'  : best_t,
        'Accuracy'   : round(test_acc * 100, 2),
        'F1_weighted': round(test_f1, 4),
        'Prec_Fake'  : round(prec_fake, 4),
        'Rec_Fake'   : round(rec_fake,  4),
        'F1_Fake'    : round(f1_fake,   4),
        'Prec_Real'  : round(prec_real, 4),
        'Rec_Real'   : round(rec_real,  4),
        'F1_Real'    : round(f1_real,   4),
    })

    print(f"\n  {name}")
    print(f"    Acc:{test_acc*100:.2f}%  F1:{test_f1:.4f}  "
          f"F1_Fake:{f1_fake:.4f}  F1_Real:{f1_real:.4f}  τ:{best_t:.2f}")


## Cell 15: Results Table

In [ ]:
results_df = pd.DataFrame(final_results)

base_acc = results_df.loc[results_df['Variant']=='A0_TITANFND_FULL','Accuracy'].values[0]
base_f1  = results_df.loc[results_df['Variant']=='A0_TITANFND_FULL','F1_weighted'].values[0]

results_df['Δ_Acc'] = (results_df['Accuracy']    - base_acc).round(2)
results_df['Δ_F1']  = (results_df['F1_weighted'] - base_f1 ).round(4)

print('\n\n' + '='*80)
print('ABLATION STUDY — COMPLETE RESULTS TABLE')
print('='*80)
print(results_df[['Variant','Accuracy','F1_weighted','F1_Fake',
                  'F1_Real','Rec_Fake','Δ_Acc','Δ_F1']].to_string(index=False))
print('='*80)

results_df.to_csv(f'{OUT_DIR}/ablation_results_welfake.csv', index=False)
print(f'\nSaved: {OUT_DIR}/ablation_results_welfake.csv')


## Cell 16: Plot 1 — Main Comparison Bar Chart

In [ ]:
LABELS   = [r['Variant'].replace('A0_TITANFND_FULL','A0\n★TITAN-FND')
              .replace('_','\n') for r in final_results]
SHORT    = [r['Variant'].split('_',1)[0] for r in final_results]
COLORS   = ['#2ecc71' if r['Variant']=='A0_TITANFND_FULL'
            else '#e74c3c' for r in final_results]

accs     = [r['Accuracy']    for r in final_results]
f1s      = [r['F1_weighted'] for r in final_results]
f1_fakes = [r['F1_Fake']     for r in final_results]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('TITAN-FND Ablation Study — WELFake Dataset',
             fontsize=15, fontweight='bold')

def bar_plot(ax, values, title, ylabel, baseline, fmt='{:.3f}'):
    bars = ax.bar(range(len(SHORT)), values, color=COLORS,
                  edgecolor='white', linewidth=1.2, width=0.65)
    ax.axhline(baseline, color='#27ae60', linestyle='--', linewidth=2,
               label=f'TITAN-FND = {baseline:{fmt[2:-1]}}')
    ax.set_xticks(range(len(SHORT)))
    ax.set_xticklabels(SHORT, fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002,
                fmt.format(val), ha='center', va='bottom', fontsize=8)

bar_plot(axes[0], accs,     'Test Accuracy (%)',   'Accuracy (%)',    base_acc, '{:.2f}')
bar_plot(axes[1], f1s,      'Weighted F1',         'F1 Score',        base_f1,  '{:.4f}')
bar_plot(axes[2], f1_fakes, 'Fake Class F1',       'F1 (Fake class)', 
         results_df.loc[results_df['Variant']=='A0_TITANFND_FULL','F1_Fake'].values[0],
         '{:.4f}')

axes[0].set_ylim(max(0, min(accs) - 3), min(100, max(accs) + 3))
axes[1].set_ylim(max(0, min(f1s)  - 0.03), min(1, max(f1s)  + 0.03))
axes[2].set_ylim(max(0, min(f1_fakes) - 0.03), min(1, max(f1_fakes) + 0.03))

green_p = mpatches.Patch(color='#2ecc71', label='Proposed: TITAN-FND (A0)')
red_p   = mpatches.Patch(color='#e74c3c', label='Ablated variants (A1–A8)')
fig.legend(handles=[green_p, red_p], loc='lower center',
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(f'{OUT_DIR}/ablation_comparison_welfake.png',
            dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: ablation_comparison_welfake.png')


## Cell 17: Plot 2 — ΔF1 Drop Chart

In [ ]:
delta_f1 = results_df['Δ_F1'].tolist()
colors2  = ['#27ae60' if d >= 0 else '#e74c3c' for d in delta_f1]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(range(len(SHORT)), delta_f1, color=colors2,
              edgecolor='white', linewidth=1.2)
ax.axhline(0, color='black', linewidth=1.2)
ax.set_xticks(range(len(SHORT)))
ax.set_xticklabels([r['Variant'] for r in final_results],
                    rotation=35, ha='right', fontsize=9)
ax.set_ylabel('ΔF1 vs TITAN-FND Full Model')
ax.set_title('Ablation: Performance Drop When Each Component is Removed\n'
             '(Negative bar = removing that component hurts performance)',
             fontsize=12, fontweight='bold')

for bar, val in zip(bars, delta_f1):
    ypos = bar.get_height() + 0.001 if val >= 0 else bar.get_height() - 0.003
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:+.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ablation_delta_f1_welfake.png', dpi=150)
plt.close()
print(f'Saved: ablation_delta_f1_welfake.png')


## Cell 18: Plot 3 — Training Curves

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('Training Curves — All Ablation Variants (WELFake)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (name, _, _, _) in enumerate(VARIANTS):
    h  = all_histories[name]
    ep = range(1, len(h['vl_f1']) + 1)
    ax = axes[idx]
    ax.plot(ep, h['tr_acc'], 'b-o', markersize=4, label='Train Acc')
    ax.plot(ep, h['vl_acc'], 'r-o', markersize=4, label='Val Acc')
    ax.plot(ep, h['vl_f1'],  'g--s', markersize=4, label='Val F1')
    star = ' ★' if name == 'A0_TITANFND_FULL' else ''
    ax.set_title(f'{name.replace("_"," ")}{star}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
    ax.set_ylim(0.5, 1.01); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ablation_curves_welfake.png', dpi=150)
plt.close()
print(f'Saved: ablation_curves_welfake.png')


## Cell 19: Plot 4 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
fig.suptitle('Confusion Matrices — WELFake Ablation Study',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (name, _, _, use_focal) in enumerate(VARIANTS):
    model  = trained_models[name]
    crit   = (FocalLoss(alpha=alpha_vals, gamma=2.0)
               if use_focal else nn.CrossEntropyLoss())
    best_t = tune_threshold(model, val_dl, crit, DEVICE)
    _, _, _, preds, labels, _ = evaluate(model, test_dl, crit, DEVICE, best_t)
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Fake','Real'], yticklabels=['Fake','Real'])
    star = ' ★' if name == 'A0_TITANFND_FULL' else ''
    axes[idx].set_title(f'{name}{star}\nτ={best_t:.2f}', fontsize=8)
    axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ablation_confusion_welfake.png', dpi=150)
plt.close()
print(f'Saved: ablation_confusion_welfake.png')


## Cell 20: Plot 5 — Radar Chart

In [ ]:
metrics     = ['Accuracy\n(÷100)', 'F1\nWeighted', 'F1\nFake',
               'Recall\nFake', 'F1\nReal']
metric_keys = ['Accuracy',         'F1_weighted',  'F1_Fake',
               'Rec_Fake',         'F1_Real']

N      = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
cmap   = plt.cm.get_cmap('tab10', len(final_results))

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for idx, r in enumerate(final_results):
    vals = [(r[k]/100 if k=='Accuracy' else r[k]) for k in metric_keys]
    vals = vals + [vals[0]]
    lw   = 3.0 if r['Variant'] == 'A0_TITANFND_FULL' else 1.2
    ls   = '-'  if r['Variant'] == 'A0_TITANFND_FULL' else '--'
    lbl  = r['Variant'].replace('A0_TITANFND_FULL', '★ A0_TITANFND_FULL (Proposed)')
    ax.plot(angles, vals, linewidth=lw, linestyle=ls,
            label=lbl, color=cmap(idx))
    if r['Variant'] == 'A0_TITANFND_FULL':
        ax.fill(angles, vals, alpha=0.12, color=cmap(idx))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('TITAN-FND Ablation — Multi-Metric Radar\nWELFake Dataset',
             size=13, fontweight='bold', pad=22)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.18),
          fontsize=8, framealpha=0.85)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ablation_radar_welfake.png',
            dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: ablation_radar_welfake.png')


## Cell 21: Final Summary

In [ ]:
print('\n\n' + '='*75)
print('  ABLATION STUDY — FINAL SUMMARY  (WELFake Dataset)')
print('='*75)
print(f'\n  {"Variant":<24} {"Acc%":>7} {"F1_wtd":>8} '
      f'{"F1_Fake":>8} {"F1_Real":>8} {"ΔAcc":>7} {"ΔF1":>8}')
print('  ' + '-'*73)

for r in final_results:
    row  = results_df[results_df['Variant'] == r['Variant']].iloc[0]
    star = ' ◀ PROPOSED' if r['Variant'] == 'A0_TITANFND_FULL' else ''
    print(f"  {r['Variant']:<24} {r['Accuracy']:>7.2f} "
          f"{r['F1_weighted']:>8.4f} {r['F1_Fake']:>8.4f} "
          f"{r['F1_Real']:>8.4f} {row['Δ_Acc']:>+7.2f} "
          f"{row['Δ_F1']:>+8.4f}{star}")

print('\n  ' + '='*73)
print('  Component importance (largest F1 drop when removed):')
ablated = results_df[results_df['Variant'] != 'A0_TITANFND_FULL'].copy()
ablated['abs_drop'] = ablated['Δ_F1'].abs()
for _, row in ablated.sort_values('abs_drop', ascending=False).iterrows():
    comp = row['Variant'].split('_', 1)[1]
    print(f'  Removing {comp:<22} → F1 drops by {abs(row["Δ_F1"]):.4f} '
          f'(Acc drops by {abs(row["Δ_Acc"]):.2f}%)')

print('\n  Saved files:')
for fname in ['ablation_results_welfake.csv',
              'ablation_comparison_welfake.png',
              'ablation_delta_f1_welfake.png',
              'ablation_curves_welfake.png',
              'ablation_confusion_welfake.png',
              'ablation_radar_welfake.png']:
    fpath = f'{OUT_DIR}/{fname}'
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f'  {"OK" if size>0 else "MISSING":<8} {fname} ({size/1024:.1f} KB)')
